In [1]:
import anndata as ad

In [2]:
adata = ad.read_h5ad("/home/tiantian/projects/aip-rahulgk/gutmodel/hmc_final_fixed/pretrain.h5ad")
adata

AnnData object with n_obs × n_vars = 106398 × 1514
    obs: 'drr', 'study_id', 'region', 'total_bases', 'instrument'
    var: 'taxa'

In [3]:
adata.var

,taxa
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter,Bacteria.Actinomycetota.Coriobacteriia.Corioba...
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella,Bacteria.Actinomycetota.Coriobacteriia.Corioba...
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia,Bacteria.Actinomycetota.Coriobacteriia.Corioba...
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia,Bacteria.Actinomycetota.Coriobacteriia.Corioba...
Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella,Bacteria.Bacillota.Bacilli.Erysipelotrichales....
...,...
Bacteria.Marinimicrobia (SAR406 clade).Incertae Sedis.Incertae Sedis.Incertae Sedis.Incertae Sedis,Bacteria.Marinimicrobia (SAR406 clade).Incerta...
Bacteria.Balneolota.Balneolia.Balneolales.Balneolaceae.Balneola,Bacteria.Balneolota.Balneolia.Balneolales.Baln...
Bacteria.Pseudomonadota.Gammaproteobacteria.Thiomicrospirales.Thiomicrospiraceae.Thiomicrospira,Bacteria.Pseudomonadota.Gammaproteobacteria.Th...
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Crocinitomicaceae.Brumimicrobium,Bacteria.Bacteroidota.Bacteroidia.Flavobacteri...


In [4]:
from pathlib import Path
import os

# Choose Qwen3 embedding model
MODEL_ID = "Qwen/Qwen3-Embedding-4B"

# Directory where you want HF to cache the checkpoint
HF_CACHE_DIR = Path("/scratch/tiantian/hf_cache")

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)  # all HF cache under here

print("Model:", MODEL_ID)
print("HF cache dir:", HF_CACHE_DIR)


Model: Qwen/Qwen3-Embedding-4B
HF cache dir: /scratch/tiantian/hf_cache


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    MODEL_ID,
    device="cuda",                     # use L40S
    cache_folder=str(HF_CACHE_DIR),    # ensure weights are stored here
)

print("Loaded model on:", model.device)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Loaded model on: cuda:0


In [7]:
taxa_list = adata.var["taxa"].tolist()
print("Example taxa:", taxa_list[:5])
taxa_list_genus_only = [t.split(".")[-1] for t in taxa_list]
print("Example genus-level taxa:", taxa_list_genus_only[:5])

Example taxa: ['Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter', 'Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella', 'Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia', 'Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia', 'Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella']
Example genus-level taxa: ['Tractidigestivibacter', 'Collinsella', 'Adlercreutzia', 'Senegalimassilia', 'Holdemanella']


In [8]:
embeddings = model.encode(
    taxa_list,
    batch_size=256,            # L40S can handle large batches
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings.shape

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

(1514, 2560)

In [9]:
embeddings_genus_only = model.encode(
    taxa_list_genus_only,
    batch_size=256,            # L40S can handle large batches
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings_genus_only.shape

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

(1514, 2560)

In [10]:
import numpy as np
save_dir = "/home/tiantian/projects/aip-rahulgk/gutmodel/hmc_final_fixed/"
np.save(save_dir + "qwen3_taxa_embeddings.npy", embeddings)
np.save(save_dir + "qwen3_taxa_embeddings_genus_only.npy", embeddings_genus_only)
# save taxa list
with open(save_dir + "taxa_list.txt", "w") as f:
    for t in taxa_list:
        f.write(t + "\n")
with open(save_dir + "taxa_list_genus_only.txt", "w") as f:
    for t in taxa_list_genus_only:
        f.write(t + "\n")